# 检查真实模型的独立 Engine 启动 seed
同 prompt / seed 重启 Engine 三次比较文本，不同 seed 各运行一次。
该测试不验收持久 Engine 的逐请求 seed，也不验收 hidden/latent 轨迹对齐。


In [ ]:
# ===== 配置：通常不用改 =====
PROMPT = (
    "Write one short imaginative nonsense sentence using unusual word choices. "
    "Do not explain anything; output only the sentence."
)

SAME_SEED = 123456
DIFFERENT_SEEDS = [111111, 222222, 333333]

# 建议真实推理时使用这些采样设置：
TEMPERATURE = 1.0
MAX_NEW_TOKENS = 64


## 真实模型版本
调用 `sample_once(prompt, seed)`；固定模型挂载路径，使用 GPU 0、tp=1、float16。
每次调用通过 Engine(random_seed=seed) 新建引擎并关闭，共加载模型6次。
前提：Kaggle 已提供作者自定义 SGLang/FlashInfer 依赖；普通 SGLang 不支持这些 latent 参数。缺依赖时停止，不自动安装。
保留本附件的 temperature=1.0、max_new_tokens=64，仅用于随机种子诊断，不作为正式 probing 配置。


In [ ]:
from pathlib import Path

MODEL_PATH = "/kaggle/input/models/fengjixing/llama3-2-1b-instruct-latent-sft-top10/pytorch/latent-sft/1"

def sample_once(prompt: str, seed: int) -> str:
    """Fresh single-GPU Engine per call: seed is an Engine startup seed."""
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("prompt must be a nonempty string")
    if isinstance(seed, bool) or not isinstance(seed, int) or not 0 <= seed < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32)")
    if not Path(MODEL_PATH).is_dir():
        raise FileNotFoundError(MODEL_PATH)

    # Requires the author's custom SGLang package already available in Kaggle.
    # No automatic installation, download, retry or OOM fallback.
    import sglang as sgl
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_PATH, local_files_only=True, trust_remote_code=True,
    )
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True,
    )
    if not text.rstrip().endswith("<think>"):
        text += "<think>"
    input_ids = tokenizer.encode(text, add_special_tokens=False)
    end_ids = tokenizer.encode("</think>", add_special_tokens=False)
    if not end_ids:
        raise ValueError("Tokenizer produced no latent end token")

    engine = sgl.Engine(
        model_path=MODEL_PATH,
        trust_remote_code=True,
        dtype="float16",
        kv_cache_dtype="auto",
        tp_size=1,
        base_gpu_id=0,
        random_seed=seed,
        enable_latent=True,
        latent_end_token_id=end_ids[0],
        disable_cuda_graph=True,
        disable_overlap_schedule=True,
        mem_fraction_static=0.90,
        sampling_backend="flashinfer",
        max_running_requests=1,
        log_level="error",
        skip_tokenizer_init=True,
        max_topk=10,
    )
    try:
        output = engine.generate(
            input_ids=input_ids,
            sampling_params={
                "temperature": TEMPERATURE,
                "top_p": 0.95,
                "max_new_tokens": MAX_NEW_TOKENS,
                "gumbel_softmax_temperature": 1.0,
                "noise_scale": 1.0,
                "add_noise_gumbel_softmax": True,
                "use_one_sided_gumbel_noise": False,
            },
        )
        return tokenizer.decode(output["output_ids"], skip_special_tokens=False)
    finally:
        engine.shutdown()


In [ ]:
# ===== 一键检查 =====
import hashlib

def short_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]

def show_run(label, seed, text):
    print(f"[{label}] seed={seed}  hash={short_hash(text)}")
    print(text)
    print("-" * 80)

print("TEST 1/2：同 seed 连续调用 3 次\n")
same_outputs = []
for i in range(3):
    text = str(sample_once(PROMPT, SAME_SEED))
    same_outputs.append(text)
    show_run(f"same-{i+1}", SAME_SEED, text)

same_seed_ok = len(set(same_outputs)) == 1

print("\nTEST 2/2：不同 seed 各调用 1 次\n")
different_outputs = []
for seed in DIFFERENT_SEEDS:
    text = str(sample_once(PROMPT, seed))
    different_outputs.append(text)
    show_run("different", seed, text)

different_seed_ok = len(set(different_outputs)) > 1

print("\n" + "=" * 80)
print("最终判断")
print("=" * 80)

if same_seed_ok and different_seed_ok:
    print("PASS ✅")
    print("同 seed 可复现，而且不同 seed 能改变采样结果。")
    print("本测试支持独立 Engine 启动 seed 的输出可复现；尚未验证持久 Engine 的请求级 seed。")
elif not same_seed_ok:
    print("FAIL ❌")
    print("同一个 seed 多次运行却得到不同结果。")
    print("优先怀疑：seed 没有传到实际 worker / sampler，或底层还有未固定的随机源。")
else:
    print("WARN ⚠️")
    print("同 seed 可以复现，但不同 seed 的输出仍完全相同。")
    print("可能原因：")
    print("1) 当前其实是 greedy / temperature=0 / do_sample=False；")
    print("2) seed 参数被忽略；")
    print("3) 这个 prompt 的概率分布太尖，碰巧选出了相同结果。")
    print("建议确认随机采样已开启，再把 temperature 调到 1.0~1.3 后重跑。")


## 结果范围
PASS 表示这六次独立 Engine 运行中，同 seed 的解码文本一致，且不同 seed 至少产生两种文本。
文本结果不能替代 latent 轨迹、worker RNG 消费和持久 Engine 请求级 seed 的验收。
FAIL 表示同 seed 文本不一致；WARN 表示不同 seed 文本一致，不能仅据此判断 seed 无效。
出现环境、依赖、设备或 OOM 错误时停止并保留报错，不自动降级重试。
